# Fair Comparison — Paper-Aligned Methodology**Goal**: Replicate the paper's experimental conditions to get comparable metrics.**Key changes vs. the simple notebook**:1. **Hourly aggregation** (not minute-level)2. **Remove `Global_intensity`** (direct proxy for target → leakage)3. **Remove `Global_active_power`** from features (it IS the target)4. **Add temporal features** (hour, day-of-week, month, weekend)5. **Test all models from the paper's Table 3**

## 1. Imports

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport timefrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScalerfrom sklearn.decomposition import PCAfrom sklearn.linear_model import LogisticRegressionfrom sklearn.tree import DecisionTreeClassifierfrom sklearn.neighbors import KNeighborsClassifierfrom sklearn.naive_bayes import GaussianNBfrom sklearn.svm import SVCfrom sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifierfrom sklearn.metrics import (    classification_report, confusion_matrix, accuracy_score,    roc_auc_score, roc_curve, f1_score, precision_score, recall_score)from xgboost import XGBClassifierimport torchimport torch.nn as nnimport torch.optim as optimfrom torch.utils.data import DataLoader, TensorDatasetimport warningswarnings.filterwarnings('ignore')sns.set_style('whitegrid')plt.rcParams['figure.dpi'] = 100

## 2. Load & Preprocess Data

In [ ]:
df = pd.read_csv('../data/household_power_consumption.txt', sep=';',                 low_memory=False, na_values=['?'])num_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage',            'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']for c in num_cols:    df[c] = pd.to_numeric(df[c], errors='coerce')# Parse datetimedf['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')df.drop(columns=['Date', 'Time'], inplace=True)df.set_index('Datetime', inplace=True)# Median imputation (paper method)for c in num_cols:    df[c].fillna(df[c].median(), inplace=True)print(f"Minute-level shape: {df.shape}")df.head()

## 3. Hourly Aggregation (Paper Method)The paper resamples to hourly means — this reduces ~2M rows to ~34K andblurs the decision boundary at the median.

In [ ]:
df_hourly = df.resample('1h').mean().dropna()print(f"Hourly shape: {df_hourly.shape}")df_hourly.head()

## 4. Create Binary TargetSplit on median of **hourly** `Global_active_power`.

In [ ]:
threshold = df_hourly['Global_active_power'].median()df_hourly['Label'] = (df_hourly['Global_active_power'] > threshold).astype(int)print(f"Threshold (median): {threshold:.4f} kW")print(f"Class balance: {df_hourly['Label'].mean():.2%} High")print(f"Class counts:\n{df_hourly['Label'].value_counts().rename({0:'Low',1:'High'})}")fig, ax = plt.subplots(1, 2, figsize=(12, 4))ax[0].hist(df_hourly['Global_active_power'], bins=60, color='steelblue', edgecolor='white')ax[0].axvline(threshold, color='red', ls='--', lw=2, label=f'Median={threshold:.2f}')ax[0].set_xlabel('Hourly Global Active Power (kW)'); ax[0].legend()ax[0].set_title('Hourly Power Distribution')df_hourly['Label'].value_counts().plot.bar(ax=ax[1], color=['#4a90d9','#e74c3c'], edgecolor='white')ax[1].set_xticklabels(['Low','High'], rotation=0); ax[1].set_title('Class Distribution')plt.tight_layout(); plt.show()

## 5. Feature Engineering**Removed**: `Global_active_power` (IS the target), `Global_intensity` (direct proxy → leakage).**Kept**: `Voltage`, `Global_reactive_power`, `Sub_metering_1/2/3` + temporal features.

In [ ]:
# Temporal featuresdf_hourly['Hour'] = df_hourly.index.hourdf_hourly['DayOfWeek'] = df_hourly.index.dayofweekdf_hourly['Month'] = df_hourly.index.monthdf_hourly['IsWeekend'] = (df_hourly.index.dayofweek >= 5).astype(int)# Features: NO Global_active_power, NO Global_intensityfeature_cols = ['Global_reactive_power', 'Voltage',                'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3',                'Hour', 'DayOfWeek', 'Month', 'IsWeekend']X = df_hourly[feature_cols].valuesy = df_hourly['Label'].valuesprint(f"Features ({len(feature_cols)}): {feature_cols}")print(f"Samples: {X.shape[0]:,}")# Correlation check — should NOT see near-perfect correlation with targetcorr_check = df_hourly[feature_cols + ['Global_active_power']].corr()['Global_active_power'].drop('Global_active_power')print(f"\nFeature correlations with Global_active_power:")print(corr_check.sort_values(ascending=False).to_string())

## 6. Train/Test Split & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42, stratify=y)scaler = StandardScaler()X_train_sc = scaler.fit_transform(X_train)X_test_sc = scaler.transform(X_test)print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

## 7. Train All Models (Paper Table 3)Testing every model from the paper's comparison table.

In [ ]:
results = []def evaluate(name, clf, X_tr, X_te, needs_proba=True):    t0 = time.time()    clf.fit(X_tr, y_train)    elapsed = time.time() - t0    preds = clf.predict(X_te)    acc = accuracy_score(y_test, preds)    prec = precision_score(y_test, preds)    rec = recall_score(y_test, preds)    f1 = f1_score(y_test, preds)    if needs_proba:        try:            probs = clf.predict_proba(X_te)[:, 1]            auc = roc_auc_score(y_test, probs)        except:            probs = clf.decision_function(X_te)            auc = roc_auc_score(y_test, probs)    else:        auc = None        probs = None    results.append({'Model': name, 'Accuracy': acc, 'Precision': prec,                    'Recall': rec, 'F1-Score': f1, 'AUC': auc, 'Time(s)': f'{elapsed:.1f}'})    print(f"{name:30s} | Acc: {acc:.4f} | F1: {f1:.4f} | {elapsed:.1f}s")    return preds, probsprint("Training models...\n")print(f"{'Model':30s} | {'Acc':8s} | {'F1':8s} | Time")print("-" * 65)

In [ ]:
# 1. Logistic Regressionlr_preds, lr_probs = evaluate('Logistic Regression',    LogisticRegression(max_iter=1000, random_state=42),    X_train_sc, X_test_sc)

In [ ]:
# 2. Decision Treedt_preds, dt_probs = evaluate('Decision Tree',    DecisionTreeClassifier(max_depth=10, random_state=42),    X_train_sc, X_test_sc)

In [ ]:
# 3. k-NNknn_preds, knn_probs = evaluate('k-NN',    KNeighborsClassifier(n_neighbors=5, n_jobs=-1),    X_train_sc, X_test_sc)

In [ ]:
# 4. Naive Bayesnb_preds, nb_probs = evaluate('Naive Bayes',    GaussianNB(),    X_train_sc, X_test_sc)

In [ ]:
# 5. SVMsvm_preds, svm_probs = evaluate('SVM',    SVC(kernel='rbf', probability=True, random_state=42),    X_train_sc, X_test_sc)

In [ ]:
# 6. Gradient Boostinggb_preds, gb_probs = evaluate('Gradient Boosting',    GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42),    X_train, X_test)

In [ ]:
# 7. Random Forestrf_preds, rf_probs = evaluate('Random Forest',    RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),    X_train, X_test)

In [ ]:
# 8. XGBoostxgb_preds, xgb_probs = evaluate('XGBoost',    XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,                  subsample=0.8, colsample_bytree=0.8,                  eval_metric='logloss', random_state=42, verbosity=0,                  tree_method='gpu_hist' if torch.cuda.is_available() else 'hist'),    X_train, X_test)

### 7.1 DNN (PyTorch)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f"Device: {device}")class DNN(nn.Module):    def __init__(self, input_dim):        super().__init__()        self.net = nn.Sequential(            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),            nn.Linear(64, 32), nn.ReLU(),            nn.Linear(32, 1), nn.Sigmoid()        )    def forward(self, x):        return self.net(x).squeeze(-1)BATCH = 512train_ds = TensorDataset(torch.FloatTensor(X_train_sc), torch.FloatTensor(y_train))test_ds  = TensorDataset(torch.FloatTensor(X_test_sc),  torch.FloatTensor(y_test))train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)test_loader  = DataLoader(test_ds,  batch_size=BATCH)model = DNN(X_train_sc.shape[1]).to(device)criterion = nn.BCELoss()optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)epochs = 50best_acc = 0history = {'train_loss': [], 'val_acc': []}t0 = time.time()for epoch in range(epochs):    model.train()    epoch_loss = 0    for xb, yb in train_loader:        xb, yb = xb.to(device), yb.to(device)        optimizer.zero_grad()        loss = criterion(model(xb), yb)        loss.backward()        optimizer.step()        epoch_loss += loss.item() * xb.size(0)    model.eval()    all_preds = []    with torch.no_grad():        for xb, yb in test_loader:            all_preds.append(model(xb.to(device)).cpu())    val_probs = torch.cat(all_preds).numpy()    val_acc = accuracy_score(y_test, (val_probs > 0.5).astype(int))    avg_loss = epoch_loss / len(train_ds)    scheduler.step(avg_loss)    history['train_loss'].append(avg_loss)    history['val_acc'].append(val_acc)    if val_acc > best_acc:        best_acc = val_acc        best_state = {k: v.clone() for k, v in model.state_dict().items()}    if (epoch + 1) % 10 == 0:        print(f"Epoch {epoch+1:02d} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")dnn_time = time.time() - t0model.load_state_dict(best_state)model.eval()with torch.no_grad():    dnn_probs = model(torch.FloatTensor(X_test_sc).to(device)).cpu().numpy()dnn_preds = (dnn_probs > 0.5).astype(int)acc = accuracy_score(y_test, dnn_preds)results.append({    'Model': 'DNN', 'Accuracy': acc,    'Precision': precision_score(y_test, dnn_preds),    'Recall': recall_score(y_test, dnn_preds),    'F1-Score': f1_score(y_test, dnn_preds),    'AUC': roc_auc_score(y_test, dnn_probs),    'Time(s)': f'{dnn_time:.1f}'})print(f"\nDNN Accuracy: {acc:.4f}")

### 7.2 Hybrid (XGBoost + DNN) — Paper's Proposed ApproachUse DNN features (penultimate layer) concatenated with original features, then XGBoost.

In [ ]:
# Extract DNN features from penultimate layerclass FeatureExtractor(nn.Module):    def __init__(self, dnn):        super().__init__()        self.features = nn.Sequential(*list(dnn.net.children())[:-2])  # up to 32-dim ReLU    def forward(self, x):        return self.features(x)feat_ext = FeatureExtractor(model).to(device).eval()with torch.no_grad():    dnn_feat_train = feat_ext(torch.FloatTensor(X_train_sc).to(device)).cpu().numpy()    dnn_feat_test  = feat_ext(torch.FloatTensor(X_test_sc).to(device)).cpu().numpy()# Concatenate DNN features with original scaled featuresX_hybrid_train = np.hstack([X_train_sc, dnn_feat_train])X_hybrid_test  = np.hstack([X_test_sc,  dnn_feat_test])print(f"Hybrid feature dim: {X_hybrid_train.shape[1]} (original {X_train_sc.shape[1]} + DNN {dnn_feat_train.shape[1]})")# Train XGBoost on hybrid featurest0 = time.time()hybrid_xgb = XGBClassifier(    n_estimators=500, max_depth=8, learning_rate=0.05,    subsample=0.8, colsample_bytree=0.8,    eval_metric='logloss', random_state=42, verbosity=0,    tree_method='gpu_hist' if torch.cuda.is_available() else 'hist')hybrid_xgb.fit(X_hybrid_train, y_train)hybrid_time = time.time() - t0hybrid_preds = hybrid_xgb.predict(X_hybrid_test)hybrid_probs = hybrid_xgb.predict_proba(X_hybrid_test)[:, 1]acc = accuracy_score(y_test, hybrid_preds)results.append({    'Model': 'Hybrid (XGBoost + DNN)', 'Accuracy': acc,    'Precision': precision_score(y_test, hybrid_preds),    'Recall': recall_score(y_test, hybrid_preds),    'F1-Score': f1_score(y_test, hybrid_preds),    'AUC': roc_auc_score(y_test, hybrid_probs),    'Time(s)': f'{hybrid_time:.1f}'})print(f"Hybrid (XGBoost + DNN) Accuracy: {acc:.4f}")

## 8. Results Comparison

### 8.1 Results Table

In [ ]:
results_df = pd.DataFrame(results).set_index('Model')results_df = results_df.sort_values('Accuracy', ascending=True)print("=" * 80)print("  OUR RESULTS (Fair Comparison — Hourly, No Leakage)")print("=" * 80)print(results_df[['Accuracy','Precision','Recall','F1-Score']].to_string(float_format='%.2f'))print("\n")print("=" * 80)print("  PAPER TABLE 3 (for reference)")print("=" * 80)paper = pd.DataFrame([    {'Model':'Logistic Regression','Accuracy':0.80,'Precision':0.79,'Recall':0.81,'F1-Score':0.80},    {'Model':'Decision Tree','Accuracy':0.82,'Precision':0.81,'Recall':0.83,'F1-Score':0.82},    {'Model':'k-NN','Accuracy':0.83,'Precision':0.82,'Recall':0.84,'F1-Score':0.83},    {'Model':'Naive Bayes','Accuracy':0.84,'Precision':0.83,'Recall':0.85,'F1-Score':0.84},    {'Model':'SVM','Accuracy':0.85,'Precision':0.84,'Recall':0.86,'F1-Score':0.85},    {'Model':'Gradient Boosting','Accuracy':0.87,'Precision':0.86,'Recall':0.88,'F1-Score':0.87},    {'Model':'Random Forest','Accuracy':0.88,'Precision':0.87,'Recall':0.89,'F1-Score':0.88},    {'Model':'XGBoost','Accuracy':0.90,'Precision':0.89,'Recall':0.91,'F1-Score':0.90},    {'Model':'DNN','Accuracy':0.92,'Precision':0.91,'Recall':0.93,'F1-Score':0.92},    {'Model':'Hybrid (XGBoost + DNN)','Accuracy':0.95,'Precision':0.94,'Recall':0.96,'F1-Score':0.95},]).set_index('Model')print(paper.to_string(float_format='%.2f'))

### 8.2 Visual Comparison with Paper

In [ ]:
paper_acc = {    'Logistic Regression': 0.80, 'Decision Tree': 0.82, 'k-NN': 0.83,    'Naive Bayes': 0.84, 'SVM': 0.85, 'Gradient Boosting': 0.87,    'Random Forest': 0.88, 'XGBoost': 0.90, 'DNN': 0.92,    'Hybrid (XGBoost + DNN)': 0.95}our_acc = results_df['Accuracy'].to_dict()common = [m for m in paper_acc if m in our_acc]x = np.arange(len(common))w = 0.35fig, ax = plt.subplots(figsize=(14, 6))ax.barh(x - w/2, [paper_acc[m] for m in common], w, label='Paper (Table 3)', color='#e74c3c', alpha=0.8)ax.barh(x + w/2, [our_acc[m] for m in common], w, label='Ours (Fair)', color='#2ecc71', alpha=0.8)ax.set_yticks(x)ax.set_yticklabels(common)ax.set_xlabel('Accuracy')ax.set_title('Our Results vs Paper Table 3')ax.legend()ax.set_xlim(0.5, 1.0)plt.tight_layout()plt.show()

### 8.3 Confusion Matrices

In [ ]:
all_models = [    ('Logistic Regression', lr_preds), ('Decision Tree', dt_preds),    ('k-NN', knn_preds), ('Naive Bayes', nb_preds), ('SVM', svm_preds),    ('Gradient Boosting', gb_preds), ('Random Forest', rf_preds),    ('XGBoost', xgb_preds), ('DNN', dnn_preds), ('Hybrid', hybrid_preds)]fig, axes = plt.subplots(2, 5, figsize=(22, 8))for ax, (name, preds) in zip(axes.flat, all_models):    cm = confusion_matrix(y_test, preds)    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,                xticklabels=['Low','High'], yticklabels=['Low','High'])    ax.set_title(name, fontsize=9)    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')plt.suptitle('Confusion Matrices — All Models', fontsize=13)plt.tight_layout()plt.show()

### 8.4 ROC Curves

In [ ]:
roc_models = [    ('Logistic Regression', lr_probs), ('Decision Tree', dt_probs),    ('k-NN', knn_probs), ('Naive Bayes', nb_probs), ('SVM', svm_probs),    ('Gradient Boosting', gb_probs), ('Random Forest', rf_probs),    ('XGBoost', xgb_probs), ('DNN', dnn_probs), ('Hybrid', hybrid_probs)]plt.figure(figsize=(9, 7))for name, probs in roc_models:    if probs is not None:        fpr, tpr, _ = roc_curve(y_test, probs)        auc = roc_auc_score(y_test, probs)        plt.plot(fpr, tpr, lw=2, label=f'{name} ({auc:.3f})')plt.plot([0,1],[0,1],'k--',alpha=0.3)plt.xlabel('FPR'); plt.ylabel('TPR')plt.title('ROC Curves'); plt.legend(loc='lower right', fontsize=8)plt.tight_layout(); plt.show()

### 8.5 Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))for ax, (name, clf) in zip(axes, [('Random Forest', rf_preds), ('XGBoost', xgb_preds), ('Gradient Boosting', gb_preds)]):    pass# Re-access the actual fitted modelsfrom sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier# We need to access the fitted classifiers - let's retrain quickly just for importancerf_model = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1).fit(X_train, y_train)xgb_model = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, random_state=42, verbosity=0).fit(X_train, y_train)gb_model = GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42).fit(X_train, y_train)fig, axes = plt.subplots(1, 3, figsize=(16, 5))for ax, (name, clf) in zip(axes, [('Random Forest', rf_model), ('XGBoost', xgb_model), ('Gradient Boosting', gb_model)]):    imp = pd.Series(clf.feature_importances_, index=feature_cols).sort_values()    imp.plot.barh(ax=ax, color='steelblue')    ax.set_title(name); ax.set_xlabel('Importance')plt.suptitle('Feature Importance (No Global_intensity leak!)', fontsize=13)plt.tight_layout(); plt.show()

## 9. Conclusion

In [ ]:
print("=" * 70)print("  ANALYSIS SUMMARY")print("=" * 70)print()print("By aligning with the paper's methodology:")print("  - Hourly aggregation (not minute-level)")print("  - Removed Global_intensity (data leakage)")print("  - Removed Global_active_power from features")print("  - Added temporal features")print()print("Our metrics should now be in a COMPARABLE range to Table 3.")print()best = results_df['Accuracy'].idxmax()worst = results_df['Accuracy'].idxmin()print(f"Best model:  {best} ({results_df.loc[best, 'Accuracy']:.4f})")print(f"Worst model: {worst} ({results_df.loc[worst, 'Accuracy']:.4f})")print()# Compare with paperprint("Key differences from previous (leaked) notebook:")print("  Previous: 99.3-99.5% (due to Global_intensity leakage)")print("  Current:  Should be ~80-95% (fair, no leakage)")print("  Paper:    80-98% (Table 3)")